In [ ]:
%pip install --quiet pandas pyarrow geopandas matplotlib folium branca plotly shapely

# Pipeline Brasil — geofencing + híbrido + Plotly

Versão Brasil-inteiro da pipeline geofencing, com a estratégia híbrida embutida nativamente (sem etapa de post-processing).

**Arquitetura:**

- Driver = CSV de renda IBGE (Brasil todo).
- Para cada endereço CNEFE, duas atribuições são gravadas em SQLite:
  - **`setor_cep_geo`** — `cd_setor` derivado por point-in-polygon contra a CD2022 (primário).
  - **`setor_cep_orig`** — `cd_setor` original do CNEFE (fallback).
- Na agregação, cada setor da renda recebe CEPs do `_geo` se existirem; se não, do `_orig`.
- Coluna `origem_cep` ∈ `{geofencing, cnefe_original, sem_endereco_cnefe}` para auditoria.

**Saída:** `saida_brasil_geofencing_hibrido/` (CSV, Parquet, JSON resumo, SQLite). 1 linha = 1 setor da renda.

**Visualização:** mapas Plotly (drill-down por município) + folium opcional.

**Tempo estimado** (Brasil completo, `TEST_MODE=False`): 2–6 horas, dominado pelo `sjoin` de ~150M+ endereços contra ~470k polígonos. Use `TEST_MODE=True` (1 arquivo, 500k linhas) para smoke.

## Roteiro

1. **Setup** — imports, paths, parâmetros (`UF_FILTER=None` = Brasil; `TEST_MODE` para smoke).
2. **Renda** — carrega CSV inteiro, classifica `motivo_renda`.
3. **Geofencing híbrido** — carrega geometria CD2022 (Brasil), processa cada CSV CNEFE em chunks, escreve em `setor_cep_geo` e `setor_cep_orig`.
4. **Agregação híbrida** — para cada setor da renda, monta `lista_ceps` priorizando `_geo`, fallback em `_orig`.
5. **Atributos do shapefile** — UF, município, `SITUACAO`, `CD_TIPO`.
6. **Join driver=renda** — output 1 linha por setor com `origem_cep` explícita.
7. **Estatísticas por UF** + export.
8. **Visualização Plotly** — choropleth interativo no nível de setor com drill-down por município.

## Setup

In [ ]:
from pathlib import Path
import json
import sqlite3
import time
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

import geopandas as gpd
from shapely.geometry import Point

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def log(message: str) -> None:
    print(message, flush=True)


def now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "saida_cnefe_uf").exists() and (candidate / "BR_setores_CD2022").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CNEFE_ROOT   = PROJECT_ROOT / "saida_cnefe_uf" / "extraido" / "SEM_UF"
RENDA_CSV    = PROJECT_ROOT / "Agregados_por_setores_renda_responsavel_BR_csv" / "Agregados_por_setores_renda_responsavel_BR.csv"
SHAPEFILE    = PROJECT_ROOT / "BR_setores_CD2022" / "BR_setores_CD2022.shp"

OUTPUT_DIR   = PROJECT_ROOT / "saida_brasil_geofencing_hibrido"
WORK_SQLITE  = OUTPUT_DIR / "brasil_geofencing_hibrido_work.sqlite"
OUT_CSV      = OUTPUT_DIR / "brasil_renda_geofencing_hibrido_setor_cep.csv"
OUT_PARQUET  = OUTPUT_DIR / "brasil_renda_geofencing_hibrido_setor_cep.parquet"
OUT_SUMMARY  = OUTPUT_DIR / "brasil_renda_geofencing_hibrido_setor_cep_resumo.json"

# Filtro de UF. None = Brasil inteiro. Ex.: ['SP'] para teste.
UF_FILTER = None

# TEST_MODE: True faz smoke (1 arquivo CNEFE, 500k linhas). False = rodada completa.
TEST_MODE = True
LIMIT_FILES         = 1       if TEST_MODE else None
LIMIT_ROWS_PER_FILE = 500_000 if TEST_MODE else None

CHUNKSIZE = 250_000
REBUILD_SQLITE = True
EXPORT_CSV = True
EXPORT_PARQUET = True

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"UF_FILTER     : {UF_FILTER}")
print(f"TEST_MODE     : {TEST_MODE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
UF_POR_SIGLA = {
    "RO":"11","AC":"12","AM":"13","RR":"14","PA":"15","AP":"16","TO":"17",
    "MA":"21","PI":"22","CE":"23","RN":"24","PB":"25","PE":"26","AL":"27",
    "SE":"28","BA":"29","MG":"31","ES":"32","RJ":"33","SP":"35","PR":"41",
    "SC":"42","RS":"43","MS":"50","MT":"51","GO":"52","DF":"53",
}
SIGLA_POR_UF = {v: k for k, v in UF_POR_SIGLA.items()}

REQUIRED_CNEFE_COLUMNS = ["CEP", "COD_SETOR", "LATITUDE", "LONGITUDE", "NV_GEO_COORD"]
REQUIRED_RENDA_COLUMNS = ["CD_SETOR", "V06001", "V06002", "V06003", "V06004", "V06005"]


def normalize_uf_filters(values):
    if not values:
        return None
    out = set()
    for v in values:
        t = str(v).strip().upper()
        if t in UF_POR_SIGLA:
            out.add(UF_POR_SIGLA[t])
        elif t in SIGLA_POR_UF:
            out.add(t)
        else:
            raise ValueError(f"UF invalida: {v}")
    return sorted(out)


def infer_uf_code_from_filename(path: Path):
    prefix = path.stem.split("_", 1)[0].strip().upper()
    if prefix in SIGLA_POR_UF:
        return prefix
    if prefix in UF_POR_SIGLA:
        return UF_POR_SIGLA[prefix]
    return None


def collect_cnefe_files(root: Path, uf_codes=None, limit_files=None):
    if not root.exists():
        raise FileNotFoundError(f"Pasta CNEFE nao encontrada: {root}")
    files = sorted(p for p in root.rglob("*.csv") if p.is_file())
    if uf_codes:
        files = [p for p in files if infer_uf_code_from_filename(p) in set(uf_codes)]
    if limit_files is not None:
        files = files[:limit_files]
    if not files:
        raise ValueError("Nenhum CSV CNEFE selecionado.")
    return files


def parse_br_number_series(s: pd.Series) -> pd.Series:
    norm = (
        s.fillna("").astype(str).str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(norm, errors="coerce")


def open_sqlite(p: Path) -> sqlite3.Connection:
    p.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(p)
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA synchronous = NORMAL")
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA cache_size = -200000")
    conn.execute("PRAGMA mmap_size = 30000000000")
    return conn


def reset_tables(conn: sqlite3.Connection) -> None:
    conn.executescript(
        """
        DROP TABLE IF EXISTS setor_cep_geo;
        DROP TABLE IF EXISTS setor_cep_orig;
        DROP TABLE IF EXISTS status_ingest;
        CREATE TABLE setor_cep_geo (
            cd_setor       TEXT NOT NULL,
            cep            TEXT NOT NULL,
            qtd_enderecos  INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
        CREATE TABLE setor_cep_orig (
            cd_setor       TEXT NOT NULL,
            cep            TEXT NOT NULL,
            qtd_enderecos  INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
        CREATE TABLE status_ingest (
            cnefe_file    TEXT PRIMARY KEY,
            rows_read     INTEGER NOT NULL,
            via_geometria INTEGER NOT NULL,
            via_fallback  INTEGER NOT NULL,
            finished_at   TEXT NOT NULL
        );
        """
    )
    conn.commit()

## Etapa 1 — Renda como driver

In [ ]:
def load_renda(csv_path: Path, uf_codes=None) -> pd.DataFrame:
    log(f"[renda] Lendo {csv_path}")
    df = pd.read_csv(csv_path, sep=";", dtype=str, keep_default_na=False, na_filter=False)
    missing = [c for c in REQUIRED_RENDA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes em {csv_path.name}: {missing}")

    df["cd_setor"] = df["CD_SETOR"].astype(str).str.strip()
    df = df.loc[df["cd_setor"].str.len() == 15].copy()
    if uf_codes:
        df = df.loc[df["cd_setor"].str.slice(0, 2).isin(set(uf_codes))].copy()

    vars_renda = ["V06001", "V06002", "V06003", "V06004", "V06005"]
    for v in vars_renda:
        df[f"raw_{v.lower()}"] = df[v].astype(str).str.strip()
        df[f"renda_{v.lower()}"] = parse_br_number_series(df[v])

    def classify(row):
        if pd.notna(row["renda_v06004"]):
            return "numerica"
        if row["raw_v06004"].upper() == "X":
            return "sigilo"
        return "ausente"

    df["motivo_renda"] = df.apply(classify, axis=1)
    keep = ["cd_setor", "motivo_renda"] + [f"renda_{v.lower()}" for v in vars_renda]
    out = df[keep].drop_duplicates("cd_setor", keep="last").reset_index(drop=True)
    log(f"[renda] Setores no driver: {len(out):,}")
    print(out["motivo_renda"].value_counts())
    return out


uf_codes = normalize_uf_filters(UF_FILTER)
renda_df = load_renda(RENDA_CSV, uf_codes=uf_codes)
display(renda_df.head())

## Etapa 2 — Geofencing híbrido

Carrega geometria CD2022 do Brasil inteiro (uma vez, ~30-60s). Em seguida, para cada arquivo CNEFE:

1. Stream em chunks de 250k linhas.
2. Constrói `GeoDataFrame` de pontos `(LATITUDE, LONGITUDE)`.
3. `sjoin(pontos, polígonos, predicate="within")` → `cd_setor_geo`.
4. **Grava nas duas tabelas SQLite**: `setor_cep_geo` (com `cd_setor_geo`) e `setor_cep_orig` (com `COD_SETOR` original do CNEFE).
5. Após terminar cada arquivo, registra em `status_ingest` (para resumability).

Pontos sem polígono correspondente (rarissimo) não são contados em `_geo`, mas vão para `_orig` normalmente — o fallback cuida.

In [ ]:
def load_shapefile_geometry(shapefile: Path, uf_codes=None) -> gpd.GeoDataFrame:
    log(f"[shapefile] Carregando geometria de {shapefile}")
    gdf = gpd.read_file(shapefile, columns=["CD_SETOR", "CD_UF", "geometry"])
    gdf["cd_setor"] = gdf["CD_SETOR"].astype(str).str.strip()
    gdf["cod_uf"]  = gdf["CD_UF"].astype(str).str.strip().str.zfill(2)
    log(f"[shapefile] Total Brasil: {len(gdf):,}  |  CRS: {gdf.crs}")
    if uf_codes:
        gdf = gdf.loc[gdf["cod_uf"].isin(set(uf_codes))]
    gdf = (
        gdf.loc[gdf["cd_setor"].str.len() == 15, ["cd_setor", "geometry"]]
        .reset_index(drop=True)
    )
    log(f"[shapefile] Setores apos filtro UF={uf_codes}: {len(gdf):,}")
    if len(gdf) == 0:
        raise RuntimeError("Shapefile retornou 0 setores apos filtro UF.")
    return gdf


shp_geo = load_shapefile_geometry(SHAPEFILE, uf_codes=uf_codes)
display(shp_geo.head(3))

In [ ]:
def ingest_cnefe_hibrido(conn, files, shp_geo, chunksize, limit_rows_per_file=None, skip_files=None):
    target_crs = shp_geo.crs
    skip_files = set(skip_files or [])
    stats = {"files": 0, "rows_read": 0, "via_geometria": 0, "via_fallback_orig": 0}

    for i, csv_path in enumerate(files, 1):
        if csv_path.name in skip_files:
            log(f"[skip] {csv_path.name} (ja processado)")
            continue
        log(f"[cnefe] {i}/{len(files)}: {csv_path.name}")
        t_file = time.time()
        file_rows = 0
        file_via_geo = 0
        file_via_orig = 0
        reader = pd.read_csv(
            csv_path, sep=";", dtype=str,
            usecols=REQUIRED_CNEFE_COLUMNS,
            keep_default_na=False, na_filter=False,
            chunksize=chunksize,
        )
        for chunk in reader:
            if limit_rows_per_file is not None:
                remaining = limit_rows_per_file - file_rows
                if remaining <= 0:
                    break
                if len(chunk) > remaining:
                    chunk = chunk.iloc[:remaining]
            raw = len(chunk)
            if raw == 0:
                continue

            lat = pd.to_numeric(chunk["LATITUDE"].str.replace(",", ".", regex=False), errors="coerce")
            lng = pd.to_numeric(chunk["LONGITUDE"].str.replace(",", ".", regex=False), errors="coerce")
            cep_digits = chunk["CEP"].fillna("").astype(str).str.replace(r"\D+", "", regex=True)
            cep = cep_digits.where(cep_digits.eq(""), cep_digits.str.zfill(8))
            cnefe_setor = chunk["COD_SETOR"].fillna("").astype(str).str.strip().str.slice(0, 15)

            mask = lat.notna() & lng.notna() & cep.ne("") & cnefe_setor.str.len().eq(15)
            if not mask.any():
                file_rows += raw
                continue

            # Grava setor_cep_orig SEMPRE (com cd_setor do CNEFE).
            orig_df = pd.DataFrame({
                "cd_setor": cnefe_setor[mask].values,
                "cep": cep[mask].values,
            })
            orig_grouped = (
                orig_df.groupby(["cd_setor", "cep"], sort=False)
                .size().reset_index(name="qtd_enderecos")
            )
            with conn:
                conn.executemany(
                    "INSERT INTO setor_cep_orig (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) "
                    "ON CONFLICT(cd_setor, cep) DO UPDATE SET qtd_enderecos = setor_cep_orig.qtd_enderecos + excluded.qtd_enderecos",
                    orig_grouped.itertuples(index=False, name=None),
                )

            # Geofencing → grava setor_cep_geo apenas para pontos que casaram com algum polígono.
            pts = gpd.GeoDataFrame(
                {"cep": cep[mask].values},
                geometry=gpd.points_from_xy(lng[mask], lat[mask]),
                crs=target_crs,
            )
            joined = gpd.sjoin(pts, shp_geo, how="left", predicate="within")
            has_geo = joined["cd_setor"].notna()

            if has_geo.any():
                geo_df = pd.DataFrame({
                    "cd_setor": joined.loc[has_geo, "cd_setor"].values,
                    "cep": joined.loc[has_geo, "cep"].values,
                })
                geo_grouped = (
                    geo_df.groupby(["cd_setor", "cep"], sort=False)
                    .size().reset_index(name="qtd_enderecos")
                )
                with conn:
                    conn.executemany(
                        "INSERT INTO setor_cep_geo (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) "
                        "ON CONFLICT(cd_setor, cep) DO UPDATE SET qtd_enderecos = setor_cep_geo.qtd_enderecos + excluded.qtd_enderecos",
                        geo_grouped.itertuples(index=False, name=None),
                    )

            via_g = int(has_geo.sum())
            via_o = int((~has_geo).sum())
            stats["rows_read"] += raw
            stats["via_geometria"] += via_g
            stats["via_fallback_orig"] += via_o
            file_rows += raw
            file_via_geo += via_g
            file_via_orig += via_o

            log(f"  chunk {raw:,}: via_geo={via_g:,}  via_orig={via_o:,}  (acumulado arquivo: {file_rows:,})")
            if limit_rows_per_file is not None and file_rows >= limit_rows_per_file:
                break

        stats["files"] += 1
        # Registra checkpoint.
        with conn:
            conn.execute(
                "INSERT OR REPLACE INTO status_ingest (cnefe_file, rows_read, via_geometria, via_fallback, finished_at) VALUES (?, ?, ?, ?, ?)",
                (csv_path.name, file_rows, file_via_geo, file_via_orig, now_iso()),
            )
        log(f"[done] {csv_path.name} em {time.time()-t_file:.1f}s | linhas={file_rows:,}")
    return stats


cnefe_files = collect_cnefe_files(CNEFE_ROOT, uf_codes=uf_codes, limit_files=LIMIT_FILES)
log(f"Arquivos CNEFE selecionados: {len(cnefe_files)}")
for p in cnefe_files[:5]:
    log(f"  {p.name}  ({p.stat().st_size/1e6:.0f} MB)")
if len(cnefe_files) > 5:
    log(f"  ... +{len(cnefe_files)-5} arquivos")

t0 = time.time()
with open_sqlite(WORK_SQLITE) as conn:
    if REBUILD_SQLITE:
        reset_tables(conn)
        skip_files = set()
    else:
        skip_files = {row[0] for row in conn.execute("SELECT cnefe_file FROM status_ingest").fetchall()}
        log(f"Reaproveitando SQLite. Pulando {len(skip_files)} arquivos ja processados.")

    geofencing_stats = ingest_cnefe_hibrido(
        conn, cnefe_files, shp_geo, CHUNKSIZE, LIMIT_ROWS_PER_FILE, skip_files=skip_files,
    )
    conn.execute("CREATE INDEX IF NOT EXISTS idx_geo_cd  ON setor_cep_geo(cd_setor)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_orig_cd ON setor_cep_orig(cd_setor)")
    conn.commit()

    pares_geo  = conn.execute("SELECT COUNT(*) FROM setor_cep_geo").fetchone()[0]
    pares_orig = conn.execute("SELECT COUNT(*) FROM setor_cep_orig").fetchone()[0]
log(f"\n[geofencing] Stats: {geofencing_stats}")
log(f"[geofencing] Pares em setor_cep_geo : {pares_geo:,}")
log(f"[geofencing] Pares em setor_cep_orig: {pares_orig:,}")
log(f"[geofencing] Tempo etapa 2: {(time.time()-t0)/60:.2f} min")

## Etapa 3 — Agregação híbrida

Para cada `cd_setor` no driver de renda, monta a lista de CEPs:

1. Tenta primeiro em `setor_cep_geo` (atribuição geométrica).
2. Se vazio, usa `setor_cep_orig` (CNEFE original).
3. Se ambos vazios → setor `sem_endereco_cnefe`.

In [ ]:
AGG_SQL = """
    SELECT
        cd_setor,
        COUNT(*)                    AS qtd_ceps,
        MIN(cep)                    AS cep_inicial,
        MAX(cep)                    AS cep_final,
        CASE WHEN MIN(cep) = MAX(cep) THEN MIN(cep)
             ELSE MIN(cep) || ' - ' || MAX(cep)
        END                         AS faixa_cep,
        SUM(qtd_enderecos)          AS total_enderecos,
        GROUP_CONCAT(cep, '|')      AS lista_ceps
    FROM {tabela}
    GROUP BY cd_setor
"""

with open_sqlite(WORK_SQLITE) as conn:
    agg_geo  = pd.read_sql_query(AGG_SQL.format(tabela="setor_cep_geo"), conn)
    agg_orig = pd.read_sql_query(AGG_SQL.format(tabela="setor_cep_orig"), conn)

log(f"agg_geo  : {len(agg_geo):,} setores")
log(f"agg_orig : {len(agg_orig):,} setores")

# Filtrar geo apenas para setores válidos (15 dígitos no padrão CD2022, e dentro do filtro UF se houver).
if uf_codes:
    valid_uf_prefix = set(uf_codes)
    agg_geo  = agg_geo.loc[agg_geo["cd_setor"].str.slice(0,2).isin(valid_uf_prefix)]
    agg_orig = agg_orig.loc[agg_orig["cd_setor"].str.slice(0,2).isin(valid_uf_prefix)]

setores_em_geo = set(agg_geo["cd_setor"])
agg_orig_only = agg_orig.loc[~agg_orig["cd_setor"].isin(setores_em_geo)].copy()

agg_geo["origem_cep"]      = "geofencing"
agg_orig_only["origem_cep"] = "cnefe_original"

setor_ceps = pd.concat([agg_geo, agg_orig_only], ignore_index=True)
log(f"Setores agregados (geo + fallback orig): {len(setor_ceps):,}")
print(setor_ceps["origem_cep"].value_counts())
display(setor_ceps.head())

## Etapa 4 — Atributos do shapefile

In [ ]:
shp_raw = gpd.read_file(
    SHAPEFILE, ignore_geometry=True,
    columns=["CD_SETOR","CD_UF","NM_UF","CD_MUN","NM_MUN","SITUACAO","CD_TIPO","AREA_KM2"],
)
shp_df = pd.DataFrame({
    "cd_setor":      shp_raw["CD_SETOR"].fillna("").astype(str).str.strip(),
    "cod_uf":        shp_raw["CD_UF"].fillna("").astype(str).str.zfill(2),
    "nm_uf":         shp_raw["NM_UF"].fillna("").astype(str).str.strip(),
    "cod_municipio": shp_raw["CD_MUN"].fillna("").astype(str).str.zfill(7),
    "nm_municipio":  shp_raw["NM_MUN"].fillna("").astype(str).str.strip(),
    "situacao":      shp_raw["SITUACAO"].fillna("").astype(str).str.strip(),
    "cd_tipo":       shp_raw["CD_TIPO"].fillna("").astype(str).str.strip(),
    "area_km2":      pd.to_numeric(shp_raw["AREA_KM2"], errors="coerce"),
})
shp_df = shp_df.loc[shp_df["cd_setor"].str.len() == 15].drop_duplicates("cd_setor")
if uf_codes:
    shp_df = shp_df.loc[shp_df["cod_uf"].isin(set(uf_codes))].reset_index(drop=True)
log(f"Setores no shapefile (UF filtrada): {len(shp_df):,}")

## Etapa 5 — Join driver=renda

In [ ]:
out = renda_df.merge(setor_ceps, on="cd_setor", how="left")
out["tem_cep"] = out["qtd_ceps"].notna().astype(int)
out["qtd_ceps"]        = out["qtd_ceps"].fillna(0).astype("int64")
out["total_enderecos"] = out["total_enderecos"].fillna(0).astype("int64")
out["origem_cep"] = out["origem_cep"].fillna("sem_endereco_cnefe")

out = out.merge(shp_df, on="cd_setor", how="left")
out["sigla_uf"] = out["cd_setor"].str.slice(0, 2).map(SIGLA_POR_UF)
out["esta_no_shapefile"] = out["cod_uf"].notna().astype(int)

final_columns = [
    "cd_setor", "sigla_uf", "cod_uf", "nm_uf",
    "cod_municipio", "nm_municipio", "situacao", "cd_tipo", "area_km2",
    "motivo_renda",
    "renda_v06001","renda_v06002","renda_v06003","renda_v06004","renda_v06005",
    "origem_cep", "tem_cep", "qtd_ceps",
    "cep_inicial", "cep_final", "faixa_cep",
    "total_enderecos", "lista_ceps", "esta_no_shapefile",
]
out = out[final_columns]
log(f"Linhas no output (1 por setor): {len(out):,}")
display(out.head())

## Etapa 6 — Estatísticas por UF

In [ ]:
summary_row = pd.DataFrame({
    "métrica": [
        "setores no driver (renda)",
        "com CEP (qualquer origem)",
        "  via geofencing",
        "  via cnefe_original (fallback)",
        "sem_endereco_cnefe",
        "renda numerica",
        "renda sigilo (X)",
        "renda ausente",
        "total enderecos",
    ],
    "valor": [
        len(out),
        int(out["tem_cep"].sum()),
        int((out["origem_cep"]=="geofencing").sum()),
        int((out["origem_cep"]=="cnefe_original").sum()),
        int((out["origem_cep"]=="sem_endereco_cnefe").sum()),
        int((out["motivo_renda"]=="numerica").sum()),
        int((out["motivo_renda"]=="sigilo").sum()),
        int((out["motivo_renda"]=="ausente").sum()),
        int(out["total_enderecos"].sum()),
    ],
})
display(summary_row)

uf_stats = (
    out.groupby("sigla_uf")
    .agg(setores=("cd_setor","count"),
         com_cep=("tem_cep","sum"),
         via_geo=("origem_cep", lambda s: (s=="geofencing").sum()),
         via_orig=("origem_cep", lambda s: (s=="cnefe_original").sum()),
         sem_cep=("origem_cep", lambda s: (s=="sem_endereco_cnefe").sum()))
    .assign(pct_cep=lambda d: (d["com_cep"]/d["setores"]*100).round(2))
    .sort_values("setores", ascending=False)
)
print("Cobertura por UF:")
display(uf_stats)

## Etapa 7 — Exportar

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if EXPORT_CSV:
    out.to_csv(OUT_CSV, sep=";", index=False, encoding="utf-8")
    log(f"[export] CSV: {OUT_CSV}")
if EXPORT_PARQUET:
    out.to_parquet(OUT_PARQUET, index=False)
    log(f"[export] Parquet: {OUT_PARQUET}")

resumo = {
    "generated_at": now_iso(),
    "uf_filter": UF_FILTER,
    "test_mode": TEST_MODE,
    "setores_output":     int(len(out)),
    "setores_com_cep":    int(out["tem_cep"].sum()),
    "setores_sem_cep":    int((out["tem_cep"]==0).sum()),
    "origem_cep":         out["origem_cep"].value_counts().to_dict(),
    "motivo_renda":       out["motivo_renda"].value_counts().to_dict(),
    "total_enderecos":    int(out["total_enderecos"].sum()),
    "geofencing_stats":   geofencing_stats,
    "outputs": {
        "csv":     str(OUT_CSV)     if EXPORT_CSV     else None,
        "parquet": str(OUT_PARQUET) if EXPORT_PARQUET else None,
        "sqlite":  str(WORK_SQLITE),
        "summary": str(OUT_SUMMARY),
    },
}
OUT_SUMMARY.write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8")
log(f"[export] Resumo: {OUT_SUMMARY}")
print(json.dumps(resumo, ensure_ascii=False, indent=2)[:2000])

## Visualização — mapas interativos

Duas visões:

1. **Plotly choropleth_mapbox** — drill-down por município. Cor = renda V06004; hover mostra `origem_cep`, CEPs, qtd_endereços.
2. **Folium** — versão alternativa (com a mesma informação) caso prefira.

In [ ]:
# Filtros de visualização.
MAP_COD_MUNICIPIO = None   # ex: '3550308' (Sao Paulo). None = município com mais setores no output.
MAP_COLOR_VAR     = "renda_v06004"  # ou 'tem_cep', 'qtd_ceps' etc.

final_base = pd.read_parquet(OUT_PARQUET)
if MAP_COD_MUNICIPIO is None:
    pool = final_base.loc[final_base["esta_no_shapefile"] == 1]
    MAP_COD_MUNICIPIO = pool.groupby("cod_municipio").size().sort_values(ascending=False).index[0]
    log(f"Município escolhido automaticamente: {MAP_COD_MUNICIPIO}")

recorte = final_base.loc[final_base["cod_municipio"] == MAP_COD_MUNICIPIO].copy()
log(f"Setores no recorte: {len(recorte):,}  | com CEP: {int(recorte['tem_cep'].sum()):,}")

geo_rec = gpd.read_file(SHAPEFILE, where=f"CD_MUN = '{MAP_COD_MUNICIPIO}'")
geo_rec["cd_setor"] = geo_rec["CD_SETOR"].astype(str).str.strip()
gmap = geo_rec[["cd_setor","geometry"]].merge(recorte, on="cd_setor", how="left")
if gmap.crs is not None and gmap.crs.to_epsg() != 4326:
    gmap = gmap.to_crs(epsg=4326)
log(f"Geometrias no recorte: {len(gmap):,}")

### Mapa Plotly — choropleth por setor

In [ ]:
# Encurtar lista_ceps para hover.
def short_ceps(s):
    if s is None or (isinstance(s, float) and pd.isna(s)) or not s:
        return ""
    ceps = str(s).split("|")
    if len(ceps) <= 5:
        return ", ".join(ceps)
    return ", ".join(ceps[:5]) + f"  ... (+{len(ceps)-5})"

gmap_plot = gmap.copy()
gmap_plot["ceps_resumo"] = gmap_plot["lista_ceps"].apply(short_ceps)

# Plotly precisa que cada feature do GeoJSON tenha um 'id' que case com a coluna locations.
import json as _json
geojson = _json.loads(gmap_plot.set_index("cd_setor")[["geometry"]].to_json())
for f in geojson["features"]:
    f["id"] = f["properties"].get("cd_setor") or f["id"]

center_geom = gmap_plot.geometry.union_all().centroid

fig = px.choropleth_mapbox(
    gmap_plot,
    geojson=geojson,
    locations="cd_setor",
    color=MAP_COLOR_VAR,
    color_continuous_scale="YlOrRd",
    hover_name="cd_setor",
    hover_data={
        "cd_setor": False,
        "nm_municipio": True,
        "sigla_uf": True,
        "situacao": True,
        "motivo_renda": True,
        "renda_v06004": ":,.2f",
        "origem_cep": True,
        "tem_cep": True,
        "qtd_ceps": True,
        "faixa_cep": True,
        "total_enderecos": True,
        "ceps_resumo": True,
    },
    mapbox_style="carto-positron",
    center={"lat": center_geom.y, "lon": center_geom.x},
    zoom=11,
    opacity=0.7,
    height=720,
)
fig.update_layout(
    title=f"{MAP_COD_MUNICIPIO} — setor x {MAP_COLOR_VAR}  |  setores: {len(gmap_plot):,}",
    margin={"r":0,"t":40,"l":0,"b":0},
)
fig.show()

### Plotly — barras de cobertura por UF

Após Brasil completo, mostra o % de cobertura por UF (geofencing vs cnefe_original vs sem_cep).

In [ ]:
uf_breakdown = (
    final_base.groupby("sigla_uf")["origem_cep"]
    .value_counts(normalize=True).mul(100).round(2)
    .rename("pct").reset_index()
)
fig_uf = px.bar(
    uf_breakdown,
    x="sigla_uf", y="pct", color="origem_cep",
    color_discrete_map={
        "geofencing":          "#1f77b4",
        "cnefe_original":      "#2ca02c",
        "sem_endereco_cnefe":  "#d62728",
    },
    barmode="stack",
    title="Cobertura de CEP por UF (% setores por origem_cep)",
    height=520,
)
fig_uf.update_layout(xaxis_title="UF", yaxis_title="% setores")
fig_uf.show()

### Folium — mapa alternativo (mesmo recorte)

Mantido como referência. Borda preta = `geofencing`; azul = `cnefe_original`; vermelha = `sem_endereco_cnefe`.

In [ ]:
import folium
import branca.colormap as cm

gmap_f = gmap.copy()
gmap_f["ceps_resumo"] = gmap_f["lista_ceps"].apply(short_ceps)
for col in gmap_f.columns:
    if col == "geometry":
        continue
    if pd.api.types.is_numeric_dtype(gmap_f[col]):
        gmap_f[col] = gmap_f[col].astype(object).where(gmap_f[col].notna(), None)
    else:
        gmap_f[col] = gmap_f[col].fillna("")

renda_vals = pd.to_numeric(
    pd.Series([v for v in gmap_f["renda_v06004"] if v is not None]),
    errors="coerce",
).dropna()
if len(renda_vals):
    colormap = cm.linear.YlOrRd_09.scale(float(renda_vals.min()), float(renda_vals.max()))
    colormap.caption = "Renda média do responsável (V06004) — R$"
else:
    colormap = None

ORIGEM_STYLE = {
    "geofencing":          {"color":"black","weight":0.4},
    "cnefe_original":      {"color":"blue", "weight":1.2},
    "sem_endereco_cnefe":  {"color":"red",  "weight":2.5},
}
def style_function(feature):
    p = feature["properties"]
    r = p.get("renda_v06004")
    origem = p.get("origem_cep") or "geofencing"
    fill = "#cccccc" if r is None else (colormap(r) if colormap else "#999999")
    s = ORIGEM_STYLE.get(origem, {"color":"gray","weight":1.0})
    return {"fillColor":fill,"color":s["color"],"weight":s["weight"],"fillOpacity":0.75}

center = gmap_f.geometry.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12,
               tiles="cartodbpositron", control_scale=True)

tooltip_pairs = [
    ("cd_setor","Setor:"),("nm_municipio","Município:"),("sigla_uf","UF:"),
    ("situacao","Situação:"),("motivo_renda","Motivo renda:"),
    ("renda_v06004","Renda média (R$):"),("origem_cep","Origem CEP:"),
    ("tem_cep","Tem CEP:"),("qtd_ceps","Qtd CEPs:"),
    ("faixa_cep","Faixa CEP:"),("total_enderecos","Endereços:"),
    ("ceps_resumo","CEPs:"),
]
existing = [(f,a) for f,a in tooltip_pairs if f in gmap_f.columns]
folium.GeoJson(
    gmap_f, name="setores",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=[f for f,_ in existing], aliases=[a for _,a in existing],
        sticky=True, max_width=420,
    ),
).add_to(m)
if colormap is not None:
    colormap.add_to(m)
m

## Notas operacionais

**Para rodar Brasil definitivo:**
1. Altere na célula `paths-config`: `TEST_MODE = False` e `UF_FILTER = None`.
2. Garanta espaço em disco: SQLite intermediário pode chegar a 5–10 GB.
3. **Restart Kernel + Run All**.
4. Tempo esperado: 2–6 horas (depende muito do disco).

**Resumability:** a tabela `status_ingest` no SQLite registra cada arquivo CNEFE processado. Se a rodada cair pelo meio:
- Mude `REBUILD_SQLITE = False` na célula `paths-config`.
- Run All — vai pular os arquivos já marcados em `status_ingest` e continuar de onde parou.

**Esperado por UF:** cobertura típica ≥99,5% (com a estratégia híbrida). UFs com muitos micro-setores urbanos (SP capital, RJ capital) tendem a ter mais `cnefe_original` na composição. UFs rurais (PI, MA, AC) têm mais `sem_endereco_cnefe`.

**Pontos de atenção:**
- O ponto-no-polígono usa CRS do shapefile (SIRGAS 2000 / EPSG:4674). As coordenadas do CNEFE são compatíveis a nível submétrico.
- `NV_GEO_COORD` ≥ 3 (~1,6% dos endereços em SP) são coordenadas estimadas. Foram usadas normalmente — se quiser blindagem extra, filtre-as no `ingest_cnefe_hibrido`.